In [10]:
import datetime
from collections import defaultdict

import numpy as np
import earthkit.data as ekd
# import earthkit.regrid as ekr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient
import time

from scipy.sparse import load_npz
import logging
import pickle
import xarray as xr
import pandas as pd
import os

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [11]:
CHECKPOINT = "aifs-single-mse-1.0.ckpt"
LATLON_N320_PATH = "EKR/mir_16_linear/9533e90f8433424400ab53c7fafc87ba1a04453093311c0b5bd0b35fedc1fb83.npz"
TFM_LATLON_N320 = load_npz(LATLON_N320_PATH)
N320_LATLON_PATH = "EKR/mir_16_linear/7f0be51c7c1f522592c7639e0d3f95bcbff8a044292aa281c1e73b842736d9bf.npz"
TFM_N320_LATLON = load_npz(N320_LATLON_PATH)
ERA5_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
FULL_ERA5 = xr.open_zarr(ERA5_PATH, chunks=None)
LATITUDES = np.linspace(90, -90, 721)
LONGITUDES = np.linspace(0, 359.75, 1440)

INPUT_STATE_PATH = "input_states"
OUTPUT_STATE_PATH = "output_states"

for path in [INPUT_STATE_PATH, OUTPUT_STATE_PATH]:
    if not os.path.exists(path):
        os.makedirs(path)

In [12]:
def get_latest_IFS_data():
    logging.info("Using the latest IFS data from ECMWF OpenData")
    IFS_PARAM_SFC = [
        "10u",
        "10v",
        "2d",
        "2t",
        "msl",
        "skt",
        "sp",
        "tcw",
        "lsm",
        "z",
        "slor",
        "sdor",
    ]
    IFS_PARAM_SOIL = ["vsw", "sot"]
    IFS_PARAM_PL = ["gh", "t", "u", "v", "w", "q"]
    IFS_LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
    IFS_SOIL_LEVELS = [1, 2]

    DATE = OpendataClient().latest()
    logging.info(f"Initial date is {DATE}")

    def get_open_data(param, levelist=[]):
        fields = defaultdict(list)
        # Get the data for the current date and the previous date
        for date in [DATE - datetime.timedelta(hours=6), DATE]:
            data = ekd.from_source(
                "ecmwf-open-data", date=date, param=param, levelist=levelist
            )
            for f in data:
                # Open data is between -180 and 180, we need to shift it to 0-360
                assert f.to_numpy().shape == (721, 1440)
                values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
                # Interpolate the data to from 0.25 to N320
                # values = ekr.interpolate(
                #     values, {"grid": (0.25, 0.25)}, {"grid": "N320"}
                # )
                values = TFM_LATLON_N320 * values
                # Add the values to the list
                name = (
                    f"{f.metadata('param')}_{f.metadata('levelist')}"
                    if levelist
                    else f.metadata("param")
                )
                fields[name].append(values)

        # Create a single matrix for each parameter
        for param, values in fields.items():
            fields[param] = np.stack(values)

        return fields

    # Create empty fields dictionary
    fields = {}
    # Get the surface parameters
    fields.update(get_open_data(param=IFS_PARAM_SFC))
    # Get the soil parameters
    soil = get_open_data(param=IFS_PARAM_SOIL, levelist=IFS_SOIL_LEVELS)

    # Map the soil parameters to the expected names
    mapping = {"sot_1": "stl1", "sot_2": "stl2", "vsw_1": "swvl1", "vsw_2": "swvl2"}
    for k, v in soil.items():
        fields[mapping[k]] = v

    # Get the pressure level parameters
    fields.update(get_open_data(param=IFS_PARAM_PL, levelist=IFS_LEVELS))

    # Transform GH to Z
    for level in IFS_LEVELS:
        gh = fields.pop(f"gh_{level}")
        fields[f"z_{level}"] = gh * 9.80665

    input_state = dict(date=DATE, fields=fields)
    save_path = f"{INPUT_STATE_PATH}/input_state_{DATE.strftime('%Y%m%dT%H')}_IFS.pkl"
    # write out the input state to a file with the date in the filename
    with open(save_path, "wb") as f:
        pickle.dump(input_state, f)

    return input_state


In [13]:
def get_ERA5(init_date):
    PARAM_PL_ERA5 = [
        "geopotential",
        "temperature",
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "specific_humidity",
    ]
    PARAM_SFC_ERA5 = [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
        "2m_dewpoint_temperature",
        "mean_sea_level_pressure",
        "skin_temperature",
        "surface_pressure",
        "total_column_water",
        "land_sea_mask",
        "geopotential_at_surface",
        "sea_surface_temperature",
        "volumetric_soil_water_layer_1",
        "volumetric_soil_water_layer_2",
        "soil_temperature_level_1",
        "soil_temperature_level_2",
        "standard_deviation_of_orography",
        "slope_of_sub_gridscale_orography",
    ]
    RENAME_SFC = {
        "10m_u_component_of_wind": "10u",
        "10m_v_component_of_wind": "10v",
        "2m_temperature": "2t",
        "2m_dewpoint_temperature": "2d",
        "mean_sea_level_pressure": "msl",
        "skin_temperature": "skt",
        "surface_pressure": "sp",
        "total_column_water": "tcw",
        "land_sea_mask": "lsm",
        "geopotential_at_surface": "z",
        "sea_surface_temperature": "sst",
        "volumetric_soil_water_layer_1": "swvl1",
        "volumetric_soil_water_layer_2": "swvl2",
        "soil_temperature_level_1": "stl1",
        "soil_temperature_level_2": "stl2",
        "standard_deviation_of_orography": "sdor",
        "slope_of_sub_gridscale_orography": "slor",
    }
    RENAME_PL = {
        "geopotential": "z",
        "temperature": "t",
        "u_component_of_wind": "u",
        "v_component_of_wind": "v",
        "vertical_velocity": "w",
        "specific_humidity": "q",
    }
    LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
    PARAM_SFC = [
        "10u",
        "10v",
        "2d",
        "2t",
        "msl",
        "skt",
        "sp",
        "tcw",
        "lsm",
        "z",
        "slor",
        "sdor",
        "stl1",
        "stl2",
        "swvl1",
        "swvl2",
    ]
    PARAM_PL = ["z", "t", "u", "v", "w", "q"]

    logging.info(f"Getting ERA5 data for date {init_date}")
    init_date_minus_6 = init_date - datetime.timedelta(hours=6)

    logging.info("Getting pressure level data...")
    pl_ds = (
        FULL_ERA5[PARAM_PL_ERA5]
        .sel(time=[init_date_minus_6, init_date], level=LEVELS)
        .compute()
        .rename(RENAME_PL)
    )

    logging.info("Getting surface level data...")
    sfc_ds = (
        FULL_ERA5[PARAM_SFC_ERA5]
        .sel(time=[init_date_minus_6, init_date])
        .compute()
        .rename(RENAME_SFC)
    )

    logging.info("Processing surface level data...")
    fields_sfc = defaultdict(list)
    for date in sfc_ds.time:
        sfc_ds_date = sfc_ds.sel(time=date)
        for param in PARAM_SFC:
            values = sfc_ds_date[param].to_numpy().flatten()
            # print(values.shape)
            # values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            values = TFM_LATLON_N320 * values
            # print(values.shape)
            fields_sfc[param].append(values)

    logging.info("Processing pressure level data...")
    fields_pl = defaultdict(list)
    for date in pl_ds.time:
        pl_ds_date = pl_ds.sel(time=date)
        for param in PARAM_PL:
            for level in LEVELS:
                values = pl_ds_date[param].sel(level=level).to_numpy().flatten()
                # print(values.shape)
                # values = ekr.interpolate(
                #     values, {"grid": (0.25, 0.25)}, {"grid": "N320"}
                # )
                values = TFM_LATLON_N320 * values
                # print(values.shape)
                fields_pl[f"{param}_{level}"].append(values)

    logging.info("Making input state...")
    fields = {}
    fields.update(fields_sfc)
    fields.update(fields_pl)

    for param, values in fields.items():
        fields[param] = np.stack(values)

    input_state = dict(date=init_date, fields=fields)

    logging.info("Saving input state to file...")
    save_path = f"{INPUT_STATE_PATH}/input_state_{init_date.strftime('%Y%m%dT%H')}_ERA5.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(input_state, f)
        logging.info(f"Input state saved to {save_path}")

    logging.info(f"Input state for {init_date} created successfully.")
    return input_state

In [5]:
def process_step(output_state, runcount):
    data_vars = {}
    logging.info(f"Processing step {runcount}")
    for field in output_state['fields']:
        values = (TFM_N320_LATLON * output_state['fields'][field].reshape(-1,1)).reshape(721,1440)
        data_vars[field] = (["lat", "lon"], values.astype(np.float32))

    step_ds = xr.Dataset(
        data_vars,
        coords={"lat": LATITUDES, "lon": LONGITUDES},
    )
    step_ds = step_ds.expand_dims('step')
    step_ds['step'] = [int(runcount)]
    return step_ds

In [ ]:
# ---- PATCHED run_inference: include save_vars in output filename ----
def run_inference(init_date=None, lead_time=360, save_vars=None):
    import time, os, logging, pandas as pd, xarray as xr

    if lead_time < 6 or lead_time % 6 != 0:
        raise ValueError("Lead time must be a multiple of 6 hours and at least 6 hours.")

    if save_vars is None and lead_time > 120:
        logging.warning("Running this model for more than 120 steps and saving all variables is not recommended.")

    ic_src = "IFS" if init_date is None else "ERA5"

    # NEW: encode which vars are saved so different runs don't overwrite each other
    vars_tag = "ALL" if save_vars is None else "-".join(save_vars)
    save_path = f"{OUTPUT_STATE_PATH}/init_{ic_src}_{init_date.strftime('%Y%m%dT%H')}_lead_{lead_time}_vars_{vars_tag}.zarr"

    if os.path.exists(save_path):
        logging.info(f"Output file {save_path} already exists. Skipping inference.")
        logging.info("Loading existing dataset...")
        return xr.open_zarr(save_path)

    # build input state
    if ic_src == "IFS":
        input_state = get_latest_IFS_data()
    else:
        input_state = get_ERA5(init_date)

    runner = SimpleRunner(CHECKPOINT, device="cuda")
    current_step = 0
    start_time = time.perf_counter()
    print("Starting the inference session...")
    current_step_time = time.perf_counter()
    states = []
    for state in runner.run(input_state=input_state, lead_time=lead_time):
        print_state(state)
        current_step += 6
        if save_vars is None:
            processed_state = process_step(state, current_step)
        else:
            selected_data = {
                'date': state['date'],
                'fields': {var: state['fields'][var] for var in save_vars},
                'latitudes': state['latitudes'],
                'longitudes': state['longitudes'],
            }
            processed_state = process_step(selected_data, current_step)
        states.append(processed_state)
        logging.info(f"Step {current_step} completed.")
        step_time = time.perf_counter()
        logging.info(f"Time taken for step {current_step}: {step_time - current_step_time:.2f} s.")
        current_step_time = step_time

    logging.info("Inference session completed.")
    logging.info(f"Total time: {time.perf_counter() - start_time:.2f} s.")

    logging.info("Concatenating all steps into a single dataset.")
    ds = xr.concat(states, dim='step')
    del states
    ds = ds.expand_dims("time")
    ds["time"] = [pd.to_datetime(input_state['date'])]

    logging.info(f"Saving output dataset to {save_path}")
    ds.to_zarr(save_path, mode='w')

    return ds


In [17]:
import datetime

init_date = datetime.datetime(2023, 6, 2, 0, 0)
lead_time = 360
save_vars = ["2t", "tp", "z_500"]   # <-- add z_500

output_ds = run_inference(init_date=init_date, lead_time=lead_time, save_vars=save_vars)
print("variables in output_ds:", list(output_ds.data_vars))


2025-09-03 15:07:58,007 - INFO - Getting ERA5 data for date 2023-06-02 00:00:00
2025-09-03 15:07:58,008 - INFO - Getting pressure level data...
2025-09-03 15:20:58,172 - INFO - Getting surface level data...
2025-09-03 15:21:47,371 - INFO - Processing surface level data...
2025-09-03 15:21:47,785 - INFO - Processing pressure level data...
2025-09-03 15:21:48,525 - INFO - Making input state...
2025-09-03 15:21:48,831 - INFO - Saving input state to file...
2025-09-03 15:21:52,259 - INFO - Input state saved to input_states/input_state_20230602T00_ERA5.pkl
2025-09-03 15:21:52,260 - INFO - Input state for 2023-06-02 00:00:00 created successfully.
2025-09-03 15:21:52,264 - INFO - Using SimpleRunner runner
/opt/AIFS/lib/python3.11/site-packages/anemoi/utils/config.py:209: UserWarning: Modifying an instance of DotDict(). This class is intended to be immutable.
  warnings.warn("Modifying an instance of DotDict(). This class is intended to be immutable.")
2025-09-03 15:21:52,333 - INFO - Computed

Starting the inference session...


2025-09-03 15:21:52,496 - INFO - Expected shape for each input fields: (2, 542080)
2025-09-03 15:21:52,609 - INFO - Preparing input tensor with shape (2, 103, 542080)
2025-09-03 15:22:06,624 - INFO - Loading Checkpoint(aifs-single-mse-1.0.ckpt): 13 seconds.
2025-09-03 15:22:06,669 - INFO - Using autocast torch.float16
2025-09-03 15:22:06,670 - INFO - Lead time: 15 days, 0:00:00, time stepping: 6:00:00 Forecasting 60 steps
2025-09-03 15:22:06,671 - INFO - Forecasting step 6:00:00 (2023-06-02 06:00:00)
2025-09-03 15:22:08,579 - INFO - Processing step 6
2025-09-03 15:22:08,614 - INFO - Step 6 completed.
2025-09-03 15:22:08,615 - INFO - Time taken for step 6: 16.28 s.
2025-09-03 15:22:08,655 - INFO - Forecasting step 12:00:00 (2023-06-02 12:00:00)



😀 date=2023-06-02T06:00:00 latitudes=(542080,) longitudes=(542080,) fields=102

    q_50   shape=(542080,) min=2.05532e-06    max=3.18007e-06   
    t_1000 shape=(542080,) min=228.122        max=316.865       
    v_925  shape=(542080,) min=-34.921        max=46.5386       
    z_850  shape=(542080,) min=8118.37        max=16105.9       
    swvl2  shape=(542080,) min=0              max=0.761391      
    tcc    shape=(542080,) min=0              max=1             



2025-09-03 15:22:10,155 - INFO - Processing step 12
2025-09-03 15:22:10,174 - INFO - Step 12 completed.
2025-09-03 15:22:10,175 - INFO - Time taken for step 12: 1.56 s.
2025-09-03 15:22:10,208 - INFO - Forecasting step 18:00:00 (2023-06-02 18:00:00)


...

2025-09-03 15:23:39,619 - INFO - Processing step 360
2025-09-03 15:23:39,639 - INFO - Step 360 completed.
2025-09-03 15:23:39,640 - INFO - Time taken for step 360: 1.50 s.
2025-09-03 15:23:39,640 - INFO - Inference session completed.
2025-09-03 15:23:39,640 - INFO - Total time: 107.10 s.
2025-09-03 15:23:39,641 - INFO - Concatenating all steps into a single dataset.



😀 date=2023-06-17T00:00:00 latitudes=(542080,) longitudes=(542080,) fields=102

    q_50   shape=(542080,) min=1.88381e-06    max=3.1761e-06    
    t_1000 shape=(542080,) min=237.46         max=317.113       
    v_925  shape=(542080,) min=-32.9206       max=32.4275       
    z_850  shape=(542080,) min=8567.95        max=16212.7       
    swvl2  shape=(542080,) min=0              max=0.730344      
    tcc    shape=(542080,) min=0              max=1             



2025-09-03 15:23:39,819 - INFO - Saving output dataset to output_states/init_ERA5_20230602T00_lead_360_vars_2t-tp-z_500.zarr


variables in output_ds: ['2t', 'tp', 'z_500']


/opt/AIFS/lib/python3.11/site-packages/zarr/api/asynchronous.py:228: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [15]:
# ───────── Triptych (2t, z500, tp) — LSM coastlines, no loop/reflect, rich title, fixed colorbars ─────────
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable  # colorbars same height as panels

# AIFS var -> ERA5 var mapping
VAR_MAP = {
    "2t":    "2m_temperature",      # Kelvin
    "z_500": "geopotential",        # m^2 s^-2 (convert to meters)
    "tp":    "total_precipitation", # meters (hourly accumulation in ERA5)
}
G = 9.80665  # m s^-2

def _to_celsius(da: xr.DataArray) -> xr.DataArray:
    return (da - 273.15).assign_attrs(units="°C")

def _rename_era5_dims(da: xr.DataArray) -> xr.DataArray:
    ren = {}
    if "latitude" in da.dims:  ren["latitude"]  = "lat"
    if "longitude" in da.dims: ren["longitude"] = "lon"
    return da.rename(ren) if ren else da

def _wrap_sort_interp_to(pred_like: xr.DataArray, src: xr.DataArray) -> xr.DataArray:
    """Wrap lon to 0..360, sort lat/lon asc, interp to pred_like grid."""
    da = _rename_era5_dims(src)
    if float(da.lon.min()) < 0 or float(da.lon.max()) <= 180:
        da = da.assign_coords(lon=(da.lon % 360))
    if not np.all(np.diff(da.lon.values) > 0): da = da.sortby("lon")
    if not np.all(np.diff(da.lat.values) > 0): da = da.sortby("lat")
    da = da.interp(lat=pred_like.lat.values, lon=pred_like.lon.values)
    return da.sortby(["lat","lon"])

def _nice_var_name(var):
    return {"2t":"2m_temperature", "z_500":"z500 (m)", "tp":"total_precipitation (mm)"} \
           .get(var, var)

def _fmt_hhz(dt):
    return pd.to_datetime(dt).strftime("%Y-%m-%d %HZ")

def _fmt_step_td(hours):
    td = pd.to_timedelta(int(hours), "h")
    return f"{td.components.days:02d} days {int(td.components.hours):02d}:{int(td.components.minutes):02d}"

def make_triptych_widgets_static_refresh(
    output_ds: xr.Dataset,
    era5: xr.Dataset,
    aifs_var: str = "2t",
    use_percentile_limits: bool = True,
    diff_clip_pct: float = 99.5,
    coastlines: str = "lsm",  # "lsm" (fast), "none"
):
    if aifs_var not in output_ds:
        raise KeyError(f'"{aifs_var}" not in output_ds. Include it in save_vars.')
    if aifs_var not in VAR_MAP and aifs_var != "z_500":
        raise KeyError(f'No ERA5 mapping for "{aifs_var}". Add it to VAR_MAP.')

    # times
    init_time = pd.to_datetime(output_ds.time.values[0])
    steps = output_ds.step.values.astype(int)                       # 6, 12, ..., 360
    valid_times = [init_time + pd.to_timedelta(int(h), "h") for h in steps]

    # predictions from AIFS
    pred = output_ds[aifs_var].isel(time=0).sortby(["lat","lon"])

    # targets from ERA5 (match semantics per variable)
    if aifs_var == "z_500":
        # geopotential at 500 hPa -> meters
        tgt_raw = era5["geopotential"].sel(time=valid_times, level=500, method="nearest") / G
        tgt_raw = tgt_raw.rename("z_500")
    elif aifs_var == "tp":
        # ERA5 has hourly accumulations; sum a rolling 6-hour window that ENDS at each valid time
        tp1h_var = VAR_MAP["tp"]
        t0 = min(valid_times) - np.timedelta64(5, "h")             # need 5 hours before first valid
        t1 = max(valid_times)
        tp1h = era5[tp1h_var].sel(time=slice(t0, t1))              # hourly accumulations (m)
        # rolling sum over time (labels at window end, i.e., the valid time)
        tp6h = tp1h.rolling(time=6, min_periods=6).sum()
        tgt_raw = tp6h.sel(time=valid_times)                       # 6h accumulation ending at valid
    else:
        tgt_raw = era5[VAR_MAP[aifs_var]].sel(time=valid_times, method="nearest")

    # align to AIFS grid
    tgt = _wrap_sort_interp_to(pred, tgt_raw).assign_coords(step=("time", steps)).swap_dims(time="step")

    # land–sea mask coastlines
    lsm = None
    if coastlines == "lsm" and "land_sea_mask" in era5:
        try:
            lsm_raw = era5["land_sea_mask"].sel(time=valid_times, method="nearest")
            lsm = _wrap_sort_interp_to(pred, lsm_raw).assign_coords(step=("time", steps)).swap_dims(time="step")
        except Exception:
            lsm = None

    # unit conversions for plotting
    if aifs_var == "2t":
        pred_plot = _to_celsius(pred);  tgt_plot  = _to_celsius(tgt);  units = "°C"
    elif aifs_var == "z_500":
        pred_plot = (pred / G).assign_attrs(units="m");  tgt_plot = tgt.assign_attrs(units="m");  units = "m"
    elif aifs_var == "tp":
        pred_plot = (pred * 1000.0).assign_attrs(units="mm")   # m -> mm
        tgt_plot  = (tgt  * 1000.0).assign_attrs(units="mm")
        units = "mm"
    else:
        pred_plot = pred; tgt_plot = tgt; units = pred.attrs.get("units","")

    diff = (pred_plot - tgt_plot).astype(np.float32)

    # color limits
    if use_percentile_limits:
        pair = np.concatenate([pred_plot.values.ravel(), tgt_plot.values.ravel()])
        vmin_main = float(np.nanpercentile(pair, 0.5))
        vmax_main = float(np.nanpercentile(pair, 99.5))
        a = float(np.nanpercentile(np.abs(diff.values.ravel()), diff_clip_pct))
    else:
        vmin_main = float(min(pred_plot.min().item(), tgt_plot.min().item()))
        vmax_main = float(max(pred_plot.max().item(), tgt_plot.max().item()))
        a = float(np.nanmax(np.abs(diff.values)))
    vmin_diff, vmax_diff = -a, a

    extent = [float(pred_plot.lon.min()), float(pred_plot.lon.max()),
              float(pred_plot.lat.min()), float(pred_plot.lat.max())]

    out = widgets.Output()

    def draw(i: int):
        stp = int(steps[i])
        valid = init_time + pd.to_timedelta(stp, "h")

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.subplots_adjust(wspace=0.25)

        titles = ["Targets", "Predictions", "Diff"]
        for ax, ttl in zip(axes, titles):
            ax.set_title(ttl); ax.set_xticks([]); ax.set_yticks([])

        im0 = axes[0].imshow(tgt_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main)
        cax0 = make_axes_locatable(axes[0]).append_axes("right", size="2.5%", pad=0.08)
        c0 = fig.colorbar(im0, cax=cax0); c0.set_label(units or "")

        im1 = axes[1].imshow(pred_plot.sel(step=stp), origin="lower", extent=extent,
                             vmin=vmin_main, vmax=vmax_main)
        cax1 = make_axes_locatable(axes[1]).append_axes("right", size="2.5%", pad=0.08)
        c1 = fig.colorbar(im1, cax=cax1); c1.set_label(units or "")

        im2 = axes[2].imshow(diff.sel(step=stp), origin="lower", extent=extent,
                             cmap="RdBu_r", vmin=vmin_diff, vmax=vmax_diff)
        cax2 = make_axes_locatable(axes[2]).append_axes("right", size="2.5%", pad=0.08)
        c2 = fig.colorbar(im2, cax=cax2)
        c2.set_label(f"{aifs_var} difference ({units})" if units else f"{aifs_var} difference")

        if lsm is not None:
            for ax in axes[:3]:
                ax.contour(lsm.lon, lsm.lat, lsm.sel(step=stp), levels=[0.5],
                           colors="k", linewidths=0.4)

        title = (f"{_nice_var_name(aifs_var)}, "
                 f"init: {_fmt_hhz(init_time)}, "
                 f"valid: {_fmt_hhz(valid)}, "
                 f"Step: {_fmt_step_td(stp)}")
        fig.suptitle(title, y=1.05, fontsize=14)
        return fig

    # controls
    play    = widgets.Play(interval=400, value=0, min=0, max=len(steps)-1, step=1)
    pause   = widgets.Button(icon="stop")
    b_first = widgets.Button(icon="step-backward")
    b_prev  = widgets.Button(icon="backward")
    b_next  = widgets.Button(icon="forward")
    b_last  = widgets.Button(icon="step-forward")
    slider  = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, readout=False)
    widgets.jsdlink((play, "value"), (slider, "value"))

    def render(i):
        with out:
            out.clear_output(wait=True)
            fig = draw(i)
            display(fig)
            plt.close(fig)

    def on_slider(change):
        if change["name"] == "value":
            render(change["new"])
    def on_first(_): slider.value = slider.min
    def on_prev(_):  slider.value = max(slider.min, slider.value - 1)
    def on_next(_):  slider.value = min(slider.max, slider.value + 1)
    def on_last(_):  slider.value = slider.max
    def on_pause(_): play._playing = False

    render(0)
    slider.observe(on_slider, names="value")
    b_first.on_click(on_first); b_prev.on_click(on_prev)
    b_next.on_click(on_next);   b_last.on_click(on_last)
    pause.on_click(on_pause)

    controls = widgets.HBox([play, pause, b_first, b_prev, b_next, b_last])
    display(out); display(slider); display(controls)

# Call (examples)
#make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="2t")
#make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="z_500")
#make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="tp")   # now uses ERA5 6h sums


In [ ]:
# Default: LSM-based coastlines on the panels (Targets/Predictions/diff)
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="2t")
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="z_500")
make_triptych_widgets_static_refresh(output_ds, FULL_ERA5, aifs_var="tp")


Output()

IntSlider(value=0, max=59, readout=False)